## 1: setup

In [1]:
import sys
sys.path.append("..")

import pandas as pd

%load_ext autoreload
%autoreload 2
from src.evaluation import evaluate
from src.models import (RatingsData, make_random_recommender, make_popular_recommender,
                        make_score_recommender, itemknn_scores, svd_scores)

train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
test = pd.read_csv("../data/processed/test.csv")

## 2: refit every model on train + validation

In [2]:
history = pd.concat([train, val])
data = RatingsData(history)

models = {
    "Random": make_random_recommender(data),
    "Most popular": make_popular_recommender(data),
    "Item-kNN": make_score_recommender(data, itemknn_scores(data)),
    "SVD (15 components)": make_score_recommender(data, svd_scores(data, 15)),
}

## 3: score on the test set, once

In [3]:
rows = {}
for name, rec in models.items():
    r = evaluate(rec, history=history, target=test, k=10)
    r["coverage"] = r["n_unique_items_recommended"] / len(data.item_ids)
    rows[name] = r

final = pd.DataFrame(rows).T[
    ["precision@k", "recall@k", "ndcg@k", "coverage", "users_evaluated"]
].round(4)
print(final.to_string())
final.to_csv("../reports/final_results.csv")

                     precision@k  recall@k  ndcg@k  coverage  users_evaluated
Random                    0.0044    0.0023  0.0050    0.7964            574.0
Most popular              0.0402    0.0472  0.0592    0.0277            574.0
Item-kNN                  0.0524    0.0716  0.0730    0.1175            574.0
SVD (15 components)       0.0692    0.0913  0.0934    0.1312            574.0


## 4: how do the models do for light vs heavy users?

In [4]:
counts = history.groupby("userId").size()
groups = {
    "light users": set(counts[counts <= counts.median()].index),
    "heavy users": set(counts[counts > counts.median()].index),
}

breakdown = {}
for name, rec in models.items():
    for label, users in groups.items():
        r = evaluate(rec,
                     history=history[history["userId"].isin(users)],
                     target=test[test["userId"].isin(users)], k=10)
        breakdown[(name, label)] = {
            "precision@k": r["precision@k"], "ndcg@k": r["ndcg@k"],
            "users": r["users_evaluated"],
        }

print(pd.DataFrame(breakdown).T.round(4).to_string())

                                 precision@k  ndcg@k  users
Random              light users       0.0004  0.0003  278.0
                    heavy users       0.0064  0.0063  296.0
Most popular        light users       0.0216  0.0492  278.0
                    heavy users       0.0578  0.0686  296.0
Item-kNN            light users       0.0381  0.0761  278.0
                    heavy users       0.0659  0.0700  296.0
SVD (15 components) light users       0.0428  0.0820  278.0
                    heavy users       0.0939  0.1041  296.0


#### Findings: final test-set evaluation (k=10, 574 users)

SVD (15 components) is best on every metric: precision@10 0.0692, recall@10 0.0913, NDCG@10 0.0934, versus 0.0402 / 0.0472 / 0.0592 for most popular and 0.0524 / 0.0716 / 0.0730 for item-kNN. That is about +58% NDCG over most popular and +28% over item-kNN.

All models scored lower on test than on validation (SVD NDCG fell 20%; most popular and item-kNN, which were not tuned, fell 25-27%), so the drop mostly reflects a harder test period rather than overfitting to validation. SVD's lead over both baselines was larger on test than on validation.

SVD leads for both light and heavy users. Precision is higher for heavy users for every model, but that is partly mechanical: they have more liked movies in the test data, so a hit is easier.

Most popular has very low coverage (2.8%), recommending almost the same titles to everyone; SVD covers 13.1% of the catalogue.